# Null Value Imputation

Notebook purpose is to do some NULL value analysis and impute missing values based on what is nearby

First just import packages

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

#new library
!pip install mlxtend


In [ ]:
## import packages

import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

#NEW PACKAGES
import planetary_computer 
import dask 
from scipy import stats
from scipy.special import inv_boxcox

from datetime import datetime
from dask.distributed import Client

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.feature_selection import RFECV
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.impute import KNNImputer
import statsmodels.api as sm

import xgboost as xgb

import datetime as dt



import re


def run_groupkfold_cv(X, y, groups, n_splits=5, param_name="Parameter"):
    gkf = GroupKFold(n_splits=n_splits)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        # print(f"\n=== Fold {fold+1} ===")

        # Split
        X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]

        # Scale
        X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

        # Train
        model = train_model(X_train_scaled, y_train)

        # Evaluate (in-sample)
        y_train_pred, r2_train, rmse_t
        # Evaluate (out-sample)rain = evaluate_model(model, X_train_scaled, y_train, "Train")

        y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")

        fold_results.append((r2_train, rmse_train, r2_test, rmse_test))

    df_results_kfold = pd.DataFrame(fold_results, columns=['R2_Train', 'RMSE_Train', 'R2_Test', 'RMSE_Test']).reset_index().rename(columns={"index": "fold"})
    df_results_kfold['Parameter'] = param_name
    df_results_kfold['Features'] = ', '.join([col for col in X.columns if col != 'sample_location_group'])
    df_results_kfold = df_results_kfold[['Parameter', 'Features', 'R2_Train', 'RMSE_Train', 'R2_Test', 'RMSE_Test']]

    return df_results_kfold


In [ ]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat_features_training_combined.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)


wq_data = wq_data[['Latitude', 'Longitude', 'Sample Date', 'Red', 'Blue', 'Drad', 'Emis', 'Emsd', 'Lwir', 'Trad', 'Urad', 'Atran', 'Cdist', 'Green', 'Nir08', 'Swir16', 'Swir22', 'Atmos_Opacity']]

Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat_features_validation_combined.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v = wq_data_v[['Latitude', 'Longitude', 'Sample Date', 'Red', 'Blue', 'Drad', 'Emis', 'Emsd', 'Lwir', 'Trad', 'Urad', 'Atran', 'Cdist', 'Green', 'Nir08', 'Swir16', 'Swir22', 'Atmos_Opacity']]

In [ ]:
wq_data_nonullvalues = wq_data.dropna(how='any',axis=0)
wq_data_nullvalues = wq_data[wq_data.isnull().any(axis=1)]

wq_data_nonullvalues_v = wq_data_v.dropna(how='any',axis=0)
wq_data_nullvalues_v = wq_data_v[wq_data_v.isnull().any(axis=1)]

plt.scatter(wq_data_nonullvalues["Latitude"], wq_data_nonullvalues["Longitude"] , color='blue', label='Non-Null - Training', marker='.', s=100)

plt.scatter(wq_data_nullvalues["Latitude"], wq_data_nullvalues["Longitude"] , color='blue', label='Null - Training', marker='x')

plt.scatter(wq_data_nonullvalues_v["Latitude"], wq_data_nonullvalues_v["Longitude"] , color='red', label='Non-Null - Validation', marker='.', s=100)

plt.scatter(wq_data_nullvalues_v["Latitude"], wq_data_nullvalues_v["Longitude"] , color='red', label='Null - Validation', marker='x')

# Add labels, a title, and a legend
plt.xlabel('Latitude')
plt.ylabel('Longitude')
plt.title('Assessment of Null Values by Lat and Long')
plt.legend() # Displays the labels we set in plt.scatter()

# Display the plot
plt.show()

# Plotting
plt.figure(figsize=(10, 6))
sns.kdeplot(data=wq_data_nonullvalues['Sample Date'], fill=False, label='Training - No Null', common_norm=False, color = "Blue")
sns.kdeplot(data=wq_data_nullvalues['Sample Date'], fill=False, label='Training - Null', common_norm=False, color = "Blue", ls='--')
sns.kdeplot(data=wq_data_nonullvalues_v['Sample Date'], fill=False, label='Validation - No Null', common_norm=False, color = "Red")
sns.kdeplot(data=wq_data_nullvalues_v['Sample Date'], fill=False, label='Validation - Null', common_norm=False, color = "Red", ls='--')
plt.title('Overlayed KDE Plots (Different Sizes over Sample Date)')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=wq_data_nonullvalues['Latitude'], fill=False, label='Training - No Null', common_norm=False, color = "Blue")
sns.kdeplot(data=wq_data_nullvalues['Latitude'], fill=False, label='Training - Null', common_norm=False, color = "Blue", ls='--')
sns.kdeplot(data=wq_data_nonullvalues_v['Latitude'], fill=False, label='Validation - No Null', common_norm=False, color = "Red")
sns.kdeplot(data=wq_data_nullvalues_v['Latitude'], fill=False, label='Validation - Null', common_norm=False, color = "Red", ls='--')
plt.title('Overlayed KDE Plots (Different Sizes over Latitude)')
plt.legend()
plt.show()


plt.figure(figsize=(10, 6))
sns.kdeplot(data=wq_data_nonullvalues['Longitude'], fill=False, label='Training - No Null', common_norm=False, color = "Blue")
sns.kdeplot(data=wq_data_nullvalues['Longitude'], fill=False, label='Training - Null', common_norm=False, color = "Blue", ls='--')
sns.kdeplot(data=wq_data_nonullvalues_v['Longitude'], fill=False, label='Validation - No Null', common_norm=False, color = "Red")
sns.kdeplot(data=wq_data_nullvalues_v['Longitude'], fill=False, label='Validation - Null', common_norm=False, color = "Red", ls='--')
plt.title('Overlayed KDE Plots (Different Sizes over Longitude)')
plt.legend()
plt.show()

## Imputation Function HERE

The function that takes in the dataframe and imputes all the missing values is in the following code block. This is all that needs to be copied and pasted into an existing notebook

Best parameter choices are in the usage section below

In [ ]:
def imputevals_latlongdate(df: pd.DataFrame, nearest_neighbours, weighted, dist) -> pd.DataFrame:
    
    if not all(col in df.columns for col in ['Latitude', 'Longitude', 'Sample Date']):
        missing = [c for c in ['Latitude', 'Longitude', 'Sample Date'] if c not in df.columns]
        raise KeyError(f"Missing required columns: {missing}")

    referencedate = dt.date(2000, 1, 1)
    
    date_series = pd.to_datetime(df['Sample Date'], errors='coerce', utc=True)
    reference_date = pd.Timestamp('1970-01-01', tz='UTC')
    #  Days since the reference (as float)
    date_days = (date_series - reference_date).dt.total_seconds() / (24 * 60 * 60)
    date_days = date_days.to_numpy()

    lon = pd.to_numeric(df['Longitude'], errors='coerce').astype(float)
    lat = pd.to_numeric(df['Latitude'], errors='coerce').astype(float)

    dist_mat = np.column_stack([lon.to_numpy(), lat.to_numpy(), date_days])
    scaler = StandardScaler()
    dist_scaled = scaler.fit_transform(dist_mat)  # shape (N, 3)
    dist_scaled[:, 2] *= 0.333 #this can be tweaked - to penalize or unpenalize specific values
    
    out = df.copy()
    for c in out.columns:
        if c in ['Latitude', 'Longitude', 'Sample Date']:
            continue
        else:        
            c_vals = pd.to_numeric(out[c], errors='coerce').to_numpy(dtype=float)
            temp = np.column_stack([dist_scaled, c_vals])
            # Run KNNImputer (use distance weighting if you prefer)
            imputer = KNNImputer(n_neighbors=nearest_neighbours, weights=weighted, metric = dist)
            imputed_arr = imputer.fit_transform(temp)
            out[c] = imputed_arr[:, -1]
    return out

  
  #USAGE
  #df_nonull = imputevals_latlongdate(df, nearest_neighbours = 6, weighted = 'distance', dist='nan_euclidean')
    

## Imputation Model Diagnstics

Define a function that will run K Fold Validation against the KNN Model above to check how good it is against test data that we know the values of

In [ ]:
colstonull = ['Red', 'Blue', 'Drad', 'Emis', 'Emsd', 'Lwir', 'Trad', 'Urad', 'Atran', 'Cdist', 'Green', 'Nir08', 'Swir16', 'Swir22', 'Atmos_Opacity']
colstonull = list(dict.fromkeys(colstonull))

def runKNNoverfold(df: pd.DataFrame, kfoldgroups: int, cols_to_null=None, nearest_neighbours=None, weighted = None, dist = None):
    df_true = df.copy()

    df_cvgroup = df.copy()
    df_cvgroup['cv_group'] = np.random.randint(1, kfoldgroups + 1, size=len(df_cvgroup))

    if not all(col in df_cvgroup.columns for col in ['Latitude', 'Longitude', 'Sample Date']):
        missing = [c for c in ['Latitude', 'Longitude', 'Sample Date'] if c not in df_cvgroup.columns]
        raise KeyError(f"Missing required columns: {missing}")
    
    if cols_to_null is None:
        cols_to_null = colstonull
    
    if nearest_neighbours is None:
        nearest_neighbours = 3
        
    if weighted is None:
        weighted = 'distance'
        
    if dist is None:
        dist = 'nan_euclidean'
        
    # Prepare results container
    R2ofImputation = pd.DataFrame(columns=cols_to_null)
    
    for i in range(1, kfoldgroups+1):
        holdR2 = pd.DataFrame([{c: np.nan for c in cols_to_null}], columns=cols_to_null)
        
        mask = (df_cvgroup['cv_group'] == i)

        df_fold = df_cvgroup.copy()
        df_fold.loc[mask, cols_to_null] = np.nan

        df_impute = imputevals_latlongdate(df_fold, nearest_neighbours, weighted, dist)

        for c in cols_to_null:   
            if c not in df_true.columns or c not in df_impute.columns:
                continue
            else:
                y_true = pd.to_numeric(df_true.loc[mask,c], errors='coerce')
                y_pred = pd.to_numeric(df_impute.loc[mask, c], errors='coerce')

                R2calc = r2_score(y_true, y_pred)
                holdR2.at[0, c] = R2calc

        R2ofImputation = pd.concat([R2ofImputation,holdR2], ignore_index = True)   

    print(R2ofImputation.info())

    return R2ofImputation    
       

In [ ]:
colstonull = ['Red', 'Blue', 'Drad', 'Emis', 'Emsd', 'Lwir', 'Trad', 'Urad', 'Atran', 'Cdist', 'Green', 'Nir08', 'Swir16', 'Swir22', 'Atmos_Opacity']
colstonull = list(dict.fromkeys(colstonull))

runKNNoverfold(wq_data_nonullvalues, 5, colstonull, 6, 'distance', 'nan_euclidean') 